# Bin counting: encode a category by its target rate

_Feature Engineering_

**Replace a giant category (device ID, zip) with the click rate it has historically produced.**

Some categorical columns are enormous. In click prediction (the book's running example, on
       advertising data like the Avazu click-through set) a single column might be the device ID,
       the ad ID, or the app &mdash; with millions of distinct values. One-hot encoding
       turns that into millions of columns, almost all zero. That is wasteful and most models choke on it.

       Bin counting (also called target encoding or response-rate encoding) replaces
       the category's identity with statistics of the target for that category. Instead of a
       one-hot vector that says "this is device #8,412,067", you store one number: the historical
       click-through rate of that device, $P(\text{click}\mid \text{device})$. A million sparse
       columns collapse into a handful of dense, highly predictive ones.

---

This notebook is a practice scaffold for the **Bin counting: encode a category by its target rate** lesson. We build it up one step at a time: watch a wide, sparse one-hot matrix barely learn, then replace it with a single target-rate column — computed leakage-free — and see the test accuracy jump. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

### Step 1 — Build a high-cardinality category

Bin counting earns its keep when a category has *too many* values to one-hot. We build one: `store_id` with 800 distinct IDs, each seen only ~4 times. Each ID is secretly either "mostly negative" (10% positive) or "mostly positive" (90%), and the label is drawn from that rate — so the ID *is* the signal, but you can only learn it by counting outcomes per ID. We split into train/test **before** any encoding so the test set stays truly held out.

In [ ]:
# LESSON: Bin counting / target (response-rate) encoding — replace a HIGH-CARDINALITY
# category with the TARGET's statistics for that category (its historical positive rate),
# computed LEAKAGE-FREE (out-of-fold).
# PROBLEM: hundreds of distinct IDs, one-hot encoded, make a huge SPARSE matrix. Each ID is
# seen only a few times, so per-ID weights are noise and regularization shrinks them away.
# FIX: replace the category with its OUT-OF-FOLD target mean — ONE dense, predictive column.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(0)              # fixed seed -> reproducible numbers
n_ids = 800                                 # hundreds of categories = high cardinality
rows_per_id = 4                             # each ID seen only a handful of times
n = n_ids * rows_per_id

id_pos_rate = rng.choice([0.1, 0.9], size=n_ids)        # each ID is strongly - or strongly +
store_id = rng.integers(0, n_ids, size=n)               # each row belongs to some store
y = (rng.uniform(size=n) < id_pos_rate[store_id]).astype(int)   # label ~ that ID's rate

# Split BEFORE any encoding so the test set is truly held out.
idx = np.arange(n)
tr, te = train_test_split(idx, test_size=0.3, random_state=0, stratify=y)

### Step 2 — Reproduce the problem: one-hot is wide and sparse

One-hot encoding turns the 800 IDs into 800 columns, almost all zero. With only ~3 train rows per column, the per-ID weights are pure noise, and L2 regularization (`C=0.1`) shrinks them toward zero — washing out the real signal. We build the one-hot by hand, fit a logistic model, and record the train/test accuracy: a wide overfit gap with poor test accuracy. We also start the figure with a histogram showing how few rows each ID has.

In [ ]:
# Visualize the raw category: each ID appears only a handful of times.
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
counts = np.bincount(store_id, minlength=n_ids)
ax[0].hist(counts, bins=range(0, counts.max() + 2), color="steelblue", align="left")
ax[0].set_title(f"STEP 2: rows per store_id (median={int(np.median(counts))}) — few per ID")
ax[0].set_xlabel("# rows for a given store_id")
ax[0].set_ylabel("# of store_ids")

# Build the one-hot by hand (column j = 1 if store_id == j) to keep deps minimal.
def one_hot(ids):
    M = np.zeros((ids.shape[0], n_ids))
    M[np.arange(ids.shape[0]), ids] = 1.0
    return M

Xoh_tr = one_hot(store_id[tr])
Xoh_te = one_hot(store_id[te])

oh_clf = LogisticRegression(max_iter=2000, C=0.1)   # C=0.1: realistic L2 -> shrinks weights
oh_clf.fit(Xoh_tr, y[tr])
oh_train_acc = accuracy_score(y[tr], oh_clf.predict(Xoh_tr))
oh_acc = accuracy_score(y[te], oh_clf.predict(Xoh_te))
oh_width = Xoh_tr.shape[1]                   # 800 columns

### Step 3 — Apply the fix: out-of-fold target encoding

Now replace the ID with its positive rate. The catch is **leakage**: if a row's rate includes its own label, the feature secretly carries the answer. So for the training rows we compute the rate with K-fold — each row is encoded from the *other* folds only. **Smoothing** `(s + SM*global) / (c + SM)` shrinks rare IDs toward the global mean, a sane prior when an ID has just 1–2 rows. Test rows use the rate learned on the full train set. The scatter shows the single encoded column cleanly splitting into a low (~0.1) and high (~0.9) cluster.

In [ ]:
global_mean = y[tr].mean()
SMOOTH = 5.0                                 # pseudo-count toward the global mean

def smoothed_means(ids, labels):
    """Smoothed per-ID positive rate from (ids, labels)."""
    s = np.bincount(ids, weights=labels, minlength=n_ids)   # # positives per ID
    c = np.bincount(ids, minlength=n_ids)                   # # rows per ID
    return (s + SMOOTH * global_mean) / (c + SMOOTH)

# Honest TRAIN encoding: K-fold, each row encoded from the OTHER folds only.
enc_tr = np.empty(tr.shape[0])
kf = KFold(n_splits=5, shuffle=True, random_state=0)
for fit_i, enc_i in kf.split(tr):           # indices INTO the train block
    m = smoothed_means(store_id[tr][fit_i], y[tr][fit_i])
    enc_tr[enc_i] = m[store_id[tr][enc_i]]

# TEST encoding: rate learned from the FULL train set (test labels never touched).
m_full = smoothed_means(store_id[tr], y[tr])
enc_te = m_full[store_id[te]]

# Visualize the engineered feature: one dense column vs the actual label.
sub = rng.permutation(te.shape[0])[:600]    # subsample so the scatter stays readable
jittered_y = y[te][sub] + rng.normal(0, 0.03, len(sub))
ax[1].scatter(enc_te[sub], jittered_y, s=8, alpha=0.4)
ax[1].set_title("STEP 4: target-encoded feature (1 dense col) vs y")
ax[1].set_xlabel("out-of-fold positive-rate for this ID")
ax[1].set_ylabel("y (0/1, jittered)")
plt.tight_layout()
plt.show()

### Step 4 — Score the fix, and see why leakage is a mirage

We fit the same logistic regression on the *single* target-encoded column and compare test accuracy to the 800-column one-hot. Then a cautionary demo: a **naive in-fold** encoding (each row using its own label) looks fantastic on train but collapses on test — a reminder that the out-of-fold discipline is what makes the encoding honest.

In [ ]:
# SAME LogisticRegression on the SINGLE target-encoded column.
te_clf = LogisticRegression(max_iter=2000)
te_clf.fit(enc_tr.reshape(-1, 1), y[tr])
te_acc = accuracy_score(y[te], te_clf.predict(enc_te.reshape(-1, 1)))

# CAUTIONARY: naive IN-FOLD encoding (use each row's own label in the mean) LEAKS.
leak_enc_tr = m_full[store_id[tr]]          # encoded using the SAME rows' labels -> leak
leak_clf = LogisticRegression(max_iter=2000).fit(leak_enc_tr.reshape(-1, 1), y[tr])
leak_train_acc = accuracy_score(y[tr], leak_clf.predict(leak_enc_tr.reshape(-1, 1)))
leak_test_acc = accuracy_score(y[te], leak_clf.predict(enc_te.reshape(-1, 1)))

print(f"PROBLEM (one-hot {oh_width} cols, sparse):  train acc = {oh_train_acc:.3f}  "
      f"test acc = {oh_acc:.3f}  (overfit gap, signal shrunk away)")
print(f"FIX (out-of-fold target encoding, 1 col): test acc = {te_acc:.3f}")
print(f"  matrix width: {oh_width} cols  ->  1 col")
print(f"LEAKAGE CAVEAT — naive in-fold encoding: train acc = {leak_train_acc:.3f} "
      f"(looks great!) but test acc = {leak_test_acc:.3f}; the train score is a MIRAGE.")
print(f"PROBLEM (raw): {oh_acc:.3f}   →   FIX (engineered): {te_acc:.3f}")

## Reference implementation — pandas + numpy + scikit-learn

### Step 1 — Load the click data and set the prior

The book's example is advertising click-through data (Avazu-style), with high-cardinality ID columns like `device_id` and a binary `click` target. We load just the device ID and the label, then compute the **global click rate** — the prior we fall back to for rare or unseen IDs — and pick a smoothing strength `alpha`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

# --- Load Avazu-style click data (book example: advertising click-through) ---
# Get it from Kaggle 'Avazu CTR' / the book repo:
#   github.com/alicezheng/feature-engineering-book
# Columns include high-cardinality IDs like 'device_id' and a binary target 'click'.
df = pd.read_csv('avazu_train.csv', usecols=['device_id', 'click'])

col, target = 'device_id', 'click'
prior = df[target].mean()          # global click rate p0  (the back-off / fallback)
alpha = 20.0                       # additive-smoothing strength

### Step 2 — Plain bin counting: counts → smoothed rate → log-odds

The straightforward version groups by the category and counts clicks (`N1`) and totals (`N`) per value. Laplace smoothing `(N1 + alpha*prior) / (N + alpha)` shrinks rare values toward the prior, and taking the log-odds gives a spread-out, model-friendly version of the rate. This computes the rate over the *whole* column — shown for clarity, but for modeling you want the out-of-fold version next.

In [ ]:
# === Plain bin counting (counts -> rate -> log-odds) on the WHOLE column ===
# (Shown for clarity; for modeling use the out-of-fold version below.)
grp = df.groupby(col)[target]
counts = grp.agg(N1='sum', N=('count'))      # clicks and total per category value
counts['N0'] = counts['N'] - counts['N1']

# Conditional probability with Laplace smoothing -> shrinks rare values to the prior:
counts['rate'] = (counts['N1'] + alpha * prior) / (counts['N'] + alpha)

# Log-odds: a spread-out, model-friendly version of the rate.
counts['log_odds'] = np.log(counts['rate'] / (1.0 - counts['rate']))

### Step 3 — Out-of-fold encoding and serving-time fallback

This is the version you actually train on. K-fold splits the data, and each row's rate is computed from the *other* folds only, so no row sees its own label. Unseen values in a held-out fold fall back to the prior. Finally, for brand-new IDs at **serving** time, we look the value up in the full-train smoothed table and default to the prior when it's missing.

In [ ]:
# === Out-of-fold bin counting (the leakage-safe version you actually train on) ===
def out_of_fold_target_encode(df, col, target, alpha=20.0, n_splits=5, seed=0):
    oof = pd.Series(np.nan, index=df.index)
    prior = df[target].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(df):
        tr, val = df.iloc[tr_idx], df.iloc[val_idx]
        g = tr.groupby(col)[target].agg(N1='sum', N='count')
        rate = (g['N1'] + alpha * prior) / (g['N'] + alpha)   # smoothed rate
        # map onto the held-out fold; UNSEEN values fall back to the global rate:
        oof.iloc[val_idx] = val[col].map(rate).fillna(prior).values
    return oof

df['device_ctr'] = out_of_fold_target_encode(df, col, target, alpha=alpha)
# 'device_ctr' is now ONE dense, highly predictive column replacing a million-wide one-hot.

# For brand-new IDs at SERVING time, encode with the full-train smoothed table and
# default to the prior:
serve_rate = counts['rate']
def encode_at_serving(device_id):
    return serve_rate.get(device_id, prior)   # unseen -> global click rate p0

## Visualize the data & results

_On a real dataset, does encoding a category by its OUT-OF-FOLD target rate beat a label-encoded baseline — and how dense and predictive is that single bin-counted column?_

### Step 1 — Make a category and encode it out-of-fold

To demo on a dataset that ships with Colab, we turn the `mean radius` feature into a 5-bucket category, then target-encode those buckets with the leakage-safe out-of-fold routine (K-fold, smoothing, unseen → prior). The result `binc` is one dense column carrying each bucket's positive rate.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

d = load_breast_cancer()
ri = list(d.feature_names).index('mean radius')
x = d.data[:, ri]
y = d.target                                  # 1 = benign, 0 = malignant

# Make a 'high-cardinality category' by binning one feature into 5 buckets.
edges = np.percentile(x, [0, 20, 40, 60, 80, 100])
cat = np.clip(np.digitize(x, edges[1:-1]), 0, 4)   # category id in {0..4}

prior = y.mean()
alpha = 10.0

# --- Out-of-fold target-rate encoding (leakage-safe bin counting) ---
def oof_encode(cat, y, alpha, seed=0):
    enc = np.full(len(y), np.nan)
    skf = StratifiedKFold(5, shuffle=True, random_state=seed)
    for tr, va in skf.split(cat.reshape(-1, 1), y):
        for v in np.unique(cat[tr]):
            m = cat[tr] == v
            rate = (y[tr][m].sum() + alpha * y[tr].mean()) / (m.sum() + alpha)
            enc[va[cat[va] == v]] = rate
        enc[va] = np.where(np.isnan(enc[va]), y[tr].mean(), enc[va])  # unseen -> prior
    return enc

binc = oof_encode(cat, y, alpha)

### Step 2 — Compare label-encoded vs bin-counted accuracy

First we print each bucket's raw target rate — the dense signal, falling smoothly from ~0.95 to ~0.08. Then we cross-validate a logistic regression two ways: on the raw integer bucket id versus on the bin-counted rate column. The bin-counted version is far more accurate because the rate is monotonically related to the target, while the arbitrary integer id is not.

In [ ]:
# Per-category target rate (the dense signal):
rates = np.array([(y[cat == v].mean()) for v in range(5)])
print('per-bin target rate:', np.round(rates, 2))   # -> ~[0.95 0.86 0.71 0.33 0.08]

# Baseline: logistic regression on the raw label id vs on the bin-counted column.
acc_label = cross_val_score(LogisticRegression(max_iter=1000),
                            cat.reshape(-1, 1), y, cv=5).mean()
acc_binc  = cross_val_score(LogisticRegression(max_iter=1000),
                            binc.reshape(-1, 1), y, cv=5).mean()
print('label-encoded acc :', round(acc_label, 2))   # -> ~0.63
print('bin-counted acc   :', round(acc_binc, 2))    # -> ~0.90

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your CTR model gets 0.95 AUC in cross-validation but 0.62 AUC in production after you add a device_ctr bin-counting feature. What is the most likely cause, and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Suspect target leakage: the encoding was probably computed using each row's own label. — _If $p_v$ for a row was built from a set that includes that row, the feature partly encodes the answer, inflating CV._
- Recompute the encoding out-of-fold (or leave-one-out, or from an earlier time window). — _Each row's $device\_ctr$ must come from OTHER rows only, so no label leaks into its own prediction._
- For click models, prefer the time-split: learn rates from a back-fill window, apply forward. — _It matches how production actually works (you only ever know the past), eliminating leakage realistically._

**Answer:** Target leakage. The bin-counted rate was computed using each row's own label, so the feature secretly carried the target &mdash; brilliant in CV, useless live. Fix: compute the encoding out-of-fold / leave-one-out, or from a separate earlier time window ("back-fill"). For CTR models the time-split is the most honest because production only ever knows the past.

</details>

**Problem 2.** A device appears twice in training and both impressions were clicked, giving a raw rate of 1.0. The global click rate is $p_0=0.05$. With smoothing $\alpha=10$, what encoded value should this device get, and why is the raw $1.0$ dangerous?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note $N_1=2$, $N_0=0$, $N_v=2$ — a rare category, so the raw rate is noise. — _Two clicks out of two is a fluke; a literal $1.0$ would tell the model this device always clicks._
- Apply additive smoothing: $\hat p_v=\frac{N_1+\alpha p_0}{N_v+\alpha}=\frac{2+10\cdot 0.05}{2+10}$. — _Adding $\alpha=10$ pseudo-rows of the prior drags the tiny-count estimate toward the global rate._
- Compute: $\frac{2+0.5}{12}=\frac{2.5}{12}\approx 0.208$. — _Above average (it did click) but nowhere near $1.0$ — the confidence is appropriately low._

**Answer:** Smoothed encoding $\hat p_v=\frac{2+10\cdot 0.05}{2+10}=\frac{2.5}{12}\approx \mathbf{0.21}$. The raw $1.0$ is dangerous because two-of-two is a rare-category fluke; trusting it overfits. Additive (Laplace) smoothing backs off to the prior $p_0$, shrinking low-count estimates toward the global rate so a $1/1$ can't pose as a guaranteed click.

</details>

**Problem 3.** At serving time a request arrives with a device_id that never appeared in training. What should the bin-counting feature return, and what must you make sure your pipeline does NOT do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize this is the unseen-category case: $N_v=0$, no history for this value. — _There is no rate to compute, but the model still needs a number for this column._
- Return the global rate $p_0=P(y=1)$ as the fallback. — _It is the $N_v=0$ limit of the smoothing formula — "assume average until we learn otherwise."_
- Make sure the pipeline does not emit NaN or crash, and does not silently impute 0. — _NaN breaks the model; a hard 0 falsely claims this device never clicks, which is also wrong._

**Answer:** Return the global rate $p_0$ &mdash; the $N_v=0$ limit of $\hat p_v=\frac{N_1+\alpha p_0}{N_v+\alpha}$. The pipeline must not emit a NaN (it crashes the model) and must not silently fill 0 (that falsely claims the device never clicks). Falling back to the prior is the safe, calibrated default for any unseen value.

</details>